**Tugas 5 Praktikum Big Data (Florania Ayodya Firdyaziz)**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/17 08:52:48 WARN Utils: Your hostname, floo resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/17 08:52:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 08:52:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/17 08:53:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


**Menyiapkan Dataset**

In [4]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("TugasMandiri5") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target_cabang = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.show(5)
df_target_cabang.show()

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



[Stage 3:>                                                          (0 + 1) / 1]

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



**A. Join dan Perbandingan Target**

In [6]:
from pyspark.sql.functions import sum as spark_sum
df_total = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))
persentase_pencapaian = df_total.join(df_target_cabang, on="kota", how="inner")
persentase_pencapaian = persentase_pencapaian.withColumn("pencapaian_persen", col("total_pendapatan") / col("target_bulanan") * 100)
persentase_pencapaian = persentase_pencapaian.orderBy(col("pencapaian_persen").desc())
persentase_pencapaian.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function - Kategori Terlaris per Kota**

In [14]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc
df_kategori = df_transaksi.groupby("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))
window_kategori = Window.partitionBy("kota").orderBy(desc("total_pendapatan"))
peringkat_pendapatan = df_kategori.withColumn("peringkat", row_number().over(window_kategori))
peringkat_pendapatan = peringkat_pendapatan.filter(col("peringkat") == 1)
peringkat_pendapatan.show()

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



**C. Spark SQL**

In [10]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target_cabang.createOrReplaceTempView("target")

data_ringkasan = spark.sql("""
    SELECT
        t.kota,
        tg.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    INNER JOIN target tg
        ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

data_ringkasan.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan (Minimal 100 kata)**

Berdasarkan hasil analisis pada bagian A, cabang dengan kinerja paling baik adalah cabang Purworejo dengan total pendapatan sebanyak Rp. 45.650.000 dan pencapaian targetnya sebesar 152.1%. Hasil tersebut menunjukkan bahwa pendapatan cabang Purworejo melampaui target bulanan yang hanya Rp. 30.000.000. Sementara itu, cabang yagng paling perlu mendapatkan perhatian manajemen adalah cabang Semarang dengan total pendapatan hanya sebesar Rp. 38.175.000 dan pencapaian targetnya sekitar 63%. Persentase menunjukkan bahwa pendapatan cabang masih di bawah target bulanan. 


Berdasarkan hasil bagian B, kategori dengan pendapatan tertinggi di setiap kota juga dapat diketahui. Misalnya, di cabang Yogyakarta, kategori terlaris adalah Fashion dengan total pendapatan sebesar Rp. 13.325.000. Informasi ini dapat digunakan untuk memahami kategori produk yang memberikan pendapatan terbesar di masing-masing cabang.


Secara keseluruhan, hasil analisis join dan window function membantu manajemen membandingkan pendapatan dengan target serta mengetahui kategori yang memberi kontribusi terbesar. Data tersebut dapat digunakan sebagai bahan evaluasi dan pertimbangan dalam menentukan strategi penjualan di setiap cabang.

**Eksplorasi**

In [15]:
df_transaksi.groupBy("kota").pivot("kategori").sum("pendapatan").show()

[Stage 46:>                                                         (0 + 1) / 1]

+----------+----------+--------+----------------------+-----------------+------------+
|      kota|Elektronik| Fashion|Kesehatan & Kecantikan|Makanan & Minuman|Rumah Tangga|
+----------+----------+--------+----------------------+-----------------+------------+
|  Magelang|   7075000| 7200000|               7275000|          5225000|     4875000|
|  Semarang|   4500000| 4875000|               8475000|          9200000|    11125000|
|      Solo|   7350000| 4350000|               8425000|          7750000|     5600000|
| Purworejo|   9500000| 7575000|              10075000|          9600000|     8900000|
|Yogyakarta|  10500000|13325000|               7375000|          9200000|     6875000|
+----------+----------+--------+----------------------+-----------------+------------+



In [18]:
from pyspark.sql.functions import sum as spark_sum, round
total = df_transaksi.agg(spark_sum("pendapatan").alias("total")).collect()[0]["total"]
df_kategori = df_transaksi.groupBy("kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))
df_kategori.withColumn("kontribusi_persen", round(col("total_pendapatan") /  total * 100, 2)).orderBy(col("kontribusi_persen").desc()).show()

[Stage 53:>                                                         (0 + 1) / 1]

+--------------------+----------------+-----------------+
|            kategori|total_pendapatan|kontribusi_persen|
+--------------------+----------------+-----------------+
|Kesehatan & Kecan...|        41625000|            21.21|
|   Makanan & Minuman|        40975000|            20.88|
|          Elektronik|        38925000|            19.84|
|        Rumah Tangga|        37375000|            19.05|
|             Fashion|        37325000|            19.02|
+--------------------+----------------+-----------------+

